# Synthetic Dummy Data Generator (UMKM)
Notebook ini membuat data sintetis yang meniru pola data bisnis riil untuk fitur:

- Monthly_Revenue (IDR)
- Net_Profit_Margin (%)
- Burn_Rate_Ratio
- Transaction_Count
- Avg_Historical_Rating
- Review_Text
- Review_Volatility
- Business_Tenure_Months
- Repeat_Order_Rate (%)
- Digital_Adoption_Score
- Peak_Hour_Latency
- Location_Competitiveness

Karakteristik realistis yang dimodelkan:
- Korelasi antar fitur (contoh: adopsi digital cenderung meningkatkan repeat order dan rating)
- Trade-off operasional (kompetisi tinggi dan latency tinggi dapat menekan margin)
- Noise terkontrol agar data tidak terlalu "rapi"
- Review teks yang selaras dengan rating/sentimen

In [3]:
import random
from typing import List

import numpy as np
import pandas as pd
from faker import Faker

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

# Config
N_SAMPLES = 150000
OUTPUT_CSV = "synthetic_umkm_data.csv"

# Review templates aligned with sentiment
POSITIVE_REVIEWS = [
    "Pelayanan cepat dan ramah, pesanan selalu tepat.",
    "Kualitas produk konsisten, harga masih masuk akal.",
    "Aplikasi pemesanan mudah dipakai dan responsif.",
    "Pengiriman cepat, admin komunikatif.",
    "Selalu repeat order karena kualitasnya terjaga.",
    "Transaksi digital lancar, proses checkout tidak ribet.",
]

NEUTRAL_REVIEWS = [
    "Produk cukup baik, kadang waktu tunggu agak lama.",
    "Pelayanan standar, masih bisa ditingkatkan.",
    "Harga dan kualitas seimbang, pengalaman biasa saja.",
    "Kadang stok kosong saat jam ramai.",
    "Secara umum oke, hanya respon chat kadang lambat.",
]

NEGATIVE_REVIEWS = [
    "Pesanan sering terlambat saat jam sibuk.",
    "Kualitas tidak konsisten, kadang bagus kadang kurang.",
    "Respons admin lambat dan informasi kurang jelas.",
    "Proses pembayaran sering bermasalah.",
    "Harga naik tapi layanan tidak membaik.",
    "Sudah beberapa kali order, pengalaman makin menurun.",
]


def clamp(x: np.ndarray, low: float, high: float) -> np.ndarray:
    return np.clip(x, low, high)


def pick_review(rating: float, volatility: float, latency: str) -> str:
    """Generate review text coherent with quality signal."""
    base_pool: List[str]

    if rating >= 4.2 and volatility < 0.45 and latency == "Low":
        base_pool = POSITIVE_REVIEWS
    elif rating < 3.4 or latency == "High":
        base_pool = NEGATIVE_REVIEWS
    else:
        base_pool = NEUTRAL_REVIEWS

    text = random.choice(base_pool)

    # Add slight random variation so reviews don't look templated
    if random.random() < 0.3:
        text += " " + fake.sentence(nb_words=6)
    return text


# 1) Business maturity and competitiveness
business_tenure = np.random.randint(3, 180, size=N_SAMPLES)  # months
location_competitiveness = np.random.poisson(lam=8, size=N_SAMPLES) + 1

# 2) Digital adoption (1-10), positively related with tenure (up to a limit)
base_digital = 3.5 + 0.02 * np.sqrt(business_tenure)
noise_digital = np.random.normal(0, 1.4, N_SAMPLES)
digital_adoption = clamp(base_digital + noise_digital, 1, 10)

# 3) Transaction count depends on maturity, digital, and local competition
transaction_lambda = (
    55
    + 0.7 * business_tenure
    + 9.0 * digital_adoption
    - 2.2 * location_competitiveness
)
transaction_lambda = clamp(transaction_lambda, 20, 900)
transaction_count = np.random.poisson(transaction_lambda).astype(int)
transaction_count = np.maximum(transaction_count, 5)

# 4) Average order value (AOV) and monthly revenue
# Lognormal for realistic positive skew in monetary data
aov = np.random.lognormal(mean=np.log(65000), sigma=0.45, size=N_SAMPLES)
aov = clamp(aov, 12000, 450000)

monthly_revenue = transaction_count * aov
seasonality_noise = np.random.normal(1.0, 0.08, N_SAMPLES)
monthly_revenue = monthly_revenue * seasonality_noise
monthly_revenue = clamp(monthly_revenue, 1_500_000, 850_000_000)

# 5) Peak hour latency category influenced by transaction pressure and digital adoption
latency_score = (
    0.004 * transaction_count
    - 0.25 * digital_adoption
    + 0.08 * location_competitiveness
    + np.random.normal(0, 0.8, N_SAMPLES)
)

peak_hour_latency = np.where(
    latency_score < 0.4,
    "Low",
    np.where(latency_score < 1.5, "Med", "High")
)

# 6) Burn rate ratio (expense/revenue): worse with high competition and high latency
latency_penalty = np.select(
    [peak_hour_latency == "Low", peak_hour_latency == "Med", peak_hour_latency == "High"],
    [0.0, 0.08, 0.18],
    default=0.08,
)

burn_rate_ratio = (
    0.72
    + 0.012 * location_competitiveness
    - 0.015 * digital_adoption
    + latency_penalty
    + np.random.normal(0, 0.08, N_SAMPLES)
)
burn_rate_ratio = clamp(burn_rate_ratio, 0.45, 1.45)

# 7) Net profit margin (%), inverse relation with burn rate
net_profit_margin = (
    (1 - burn_rate_ratio) * 100
    + 0.6 * (digital_adoption - 5)
    - 0.15 * np.log1p(location_competitiveness)
    + np.random.normal(0, 2.8, N_SAMPLES)
)
net_profit_margin = clamp(net_profit_margin, -28, 42)

# 8) Repeat order rate (%), boosted by digital adoption, rating and tenure
repeat_order_rate = (
    18
    + 2.0 * digital_adoption
    + 0.035 * business_tenure
    - 0.55 * location_competitiveness
    + np.random.normal(0, 6.0, N_SAMPLES)
)
repeat_order_rate = clamp(repeat_order_rate, 4, 92)

# 9) Review volatility: higher if latency high and margin under pressure
review_volatility = (
    0.25
    + 0.18 * (peak_hour_latency == "Med").astype(float)
    + 0.34 * (peak_hour_latency == "High").astype(float)
    + 0.06 * (burn_rate_ratio > 1.0).astype(float)
    + np.random.normal(0, 0.08, N_SAMPLES)
)
review_volatility = clamp(review_volatility, 0.08, 1.25)

# 10) Average historical rating (1-5)
avg_historical_rating = (
    4.15
    + 0.07 * digital_adoption
    + 0.012 * net_profit_margin
    - 0.32 * review_volatility
    - 0.10 * (peak_hour_latency == "High").astype(float)
    + np.random.normal(0, 0.22, N_SAMPLES)
)
avg_historical_rating = clamp(avg_historical_rating, 1.0, 5.0)

# 11) Review text generation coherent with rating/volatility/latency
review_text = [
    pick_review(rating=r, volatility=v, latency=l)
    for r, v, l in zip(avg_historical_rating, review_volatility, peak_hour_latency)
]

# Final DataFrame
# Round values to more realistic reporting precision
df = pd.DataFrame(
    {
        "ID": np.arange(1, N_SAMPLES + 1),
        "Monthly_Revenue": np.round(monthly_revenue, 0).astype(int),
        "Net_Profit_Margin (%)": np.round(net_profit_margin, 2),
        "Burn_Rate_Ratio": np.round(burn_rate_ratio, 3),
        "Transaction_Count": transaction_count.astype(int),
        "Avg_Historical_Rating": np.round(avg_historical_rating, 2),
        "Review_Text": review_text,
        "Review_Volatility": np.round(review_volatility, 3),
        "Business_Tenure_Months": business_tenure.astype(int),
        "Repeat_Order_Rate (%)": np.round(repeat_order_rate, 2),
        "Digital_Adoption_Score": np.round(digital_adoption, 2),
        "Peak_Hour_Latency": peak_hour_latency,
        "Location_Competitiveness": location_competitiveness.astype(int),
    }
)

# Optional: small post-adjustment to increase realism in deficit businesses
# If burn rate is very high, cap rating and repeat order more aggressively
deficit_mask = df["Burn_Rate_Ratio"] > 1.15
df.loc[deficit_mask, "Avg_Historical_Rating"] = np.minimum(
    df.loc[deficit_mask, "Avg_Historical_Rating"],
    np.round(np.random.uniform(1.8, 3.6, deficit_mask.sum()), 2),
)
df.loc[deficit_mask, "Repeat_Order_Rate (%)"] = np.minimum(
    df.loc[deficit_mask, "Repeat_Order_Rate (%)"],
    np.round(np.random.uniform(6, 48, deficit_mask.sum()), 2),
)

# Save and preview
df.to_csv(OUTPUT_CSV, index=False)

print(f"Generated {len(df)} rows -> {OUTPUT_CSV}")
print("\nPreview:")
display(df.head(10))

print("\nSummary stats:")
display(df.describe(include="all").transpose())

Generated 150000 rows -> synthetic_umkm_data.csv

Preview:


,ID,Monthly_Revenue,Net_Profit_Margin (%),Burn_Rate_Ratio,Transaction_Count,Avg_Historical_Rating,Review_Text,Review_Volatility,Business_Tenure_Months,Repeat_Order_Rate (%),Digital_Adoption_Score,Peak_Hour_Latency,Location_Competitiveness
0,1,10746455,17.62,0.796,147,4.84,"Produk cukup baik, kadang waktu tunggu agak la...",0.365,105,22.88,4.28,Med,9
1,2,10026002,-15.23,1.059,114,3.63,"Harga dan kualitas seimbang, pengalaman biasa ...",0.516,95,11.92,1.96,Med,10
2,3,1938056,11.88,0.886,85,4.23,"Pelayanan standar, masih bisa ditingkatkan.",0.599,17,15.47,3.57,Med,8
3,4,8692714,13.52,0.837,180,4.58,"Transaksi digital lancar, proses checkout tida...",0.257,109,17.66,5.18,Low,13
4,5,3240963,-7.94,1.086,110,4.14,Harga naik tapi layanan tidak membaik. In ipsu...,0.570,74,26.87,3.05,High,7
5,6,4659352,0.45,0.975,97,4.29,Kadang stok kosong saat jam ramai. Culpa vero ...,0.573,23,23.58,5.30,Med,16
6,7,8564730,27.85,0.742,127,4.80,"Pelayanan cepat dan ramah, pesanan selalu tepa...",0.089,105,15.01,4.05,Low,6
7,8,11098374,23.78,0.748,196,4.83,Selalu repeat order karena kualitasnya terjaga.,0.279,124,36.23,6.89,Low,10
8,9,10615528,42.00,0.608,148,4.89,Selalu repeat order karena kualitasnya terjaga...,0.290,77,23.53,6.07,Low,7
9,10,9610755,6.22,0.911,144,4.33,"Secara umum oke, hanya respon chat kadang lambat.",0.469,90,28.84,3.48,Med,9



Summary stats:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,150000.0,NaN,NaN,NaN,75000.5,43301.414527,1.0,37500.75,75000.5,112500.25,150000.0
Monthly_Revenue,150000.0,NaN,NaN,NaN,9507346.17084,5618037.298234,1500000.0,5567141.5,8227221.0,11993633.75,88583609.0
Net_Profit_Margin (%),150000.0,NaN,NaN,NaN,16.939073,11.72397,-28.0,9.2,17.44,25.17,42.0
Burn_Rate_Ratio,150000.0,NaN,NaN,NaN,0.819001,0.111811,0.45,0.741,0.814,0.892,1.313
Transaction_Count,150000.0,NaN,NaN,NaN,132.144313,40.50054,27.0,100.0,132.0,164.0,271.0
Avg_Historical_Rating,150000.0,NaN,NaN,NaN,4.481031,0.323044,1.8,4.28,4.5,4.71,5.0
Review_Text,150000,45329,Selalu repeat order karena kualitasnya terjaga.,8821,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Review_Volatility,150000.0,NaN,NaN,NaN,0.352336,0.143413,0.08,0.243,0.336,0.449,0.936
Business_Tenure_Months,150000.0,NaN,NaN,NaN,91.00684,51.104736,3.0,47.0,91.0,135.0,179.0
Repeat_Order_Rate (%),150000.0,NaN,NaN,NaN,23.62019,7.042203,4.0,18.87,23.61,28.38,57.56


## Dataset Description
Dokumentasi lengkap dataset tersedia di file `README.md` pada folder yang sama.

Ringkasan isi dokumentasi:
- Tujuan dan konteks penggunaan data sintetis UMKM
- Definisi teknis tiap fitur (data dictionary)
- Logika sintesis dan asumsi hubungan antar variabel
- Karakteristik realisme, batasan dataset, dan reproducibility
- Contoh penggunaan untuk EDA, ML, dan NLP